# Summary

Load datasets to GCP BigQuery

In [1]:
import os, sys
import pandas as pd

import json


# Numantic utilities
utils_path = "/Users/stephengodfrey/Documents/Workbench/Numantic/utilities/.."
sys.path.insert(0, utils_path)
from utilities.osa_tools.authentication import ApiAuthentication

from utilities.google_tools import bigquery_tools as bqt

api_configs = ApiAuthentication(client="Numantic")



## Read local data


In [3]:
input_data_path = "../data/rag_eval_dataset"
docs_filename = "documents.csv"
multi_pas_qs = "multi_passage_answer_questions.csv"
no_answer_qs = "no_answer_questions.csv"
single_pas_answer_qs = "single_passage_answer_questions.csv"

# Read local data into Pandas dataframes
df_docs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, docs_filename))
df_mpqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, multi_pas_qs))
df_noaqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, no_answer_qs))
df_spqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, single_pas_answer_qs))

## Clean up docs

## Add some metadata for testing

In [4]:
source_type_map = {"https://enterthegungeon.fandom.com/wiki/Bullet_Kin": "gaming",
                   "https://www.dropbox.com/scl/fi/ljtdg6eaucrbf1aksw5rm/c2%20-%20session%2050%20-%20underground.docx?rlkey=ioqwgkd14i5xk20i3fp38nzgs&e=1&dl=0": "gaming",
                   "https://bytes-and-nibbles.web.app/bytes/stici-note-part-1-planning-and-prototyping": "data_science",
                   "https://github.com/llmware-ai/llmware": "data_science",
                   "https://docs.marimo.io/recipes.html": "recipes",
                   "https://towardsdatascience.com/how-to-maximize-your-impact-as-a-data-scientist-3881995a9cb1": "data_science",
                   "https://ec.europa.eu/commission/presscorner/detail/en/QANDA_21_1683": "government",
                   "https://bg3.wiki/wiki/The_Emperor": "gaming",
                   "https://whattocook.substack.com/p/so-into-northern-spain": "recipes",
                   "https://dmtalkies.com/the-zone-of-interest-ending-explained-and-summary-2023-film/": "entertainment",
                   "https://www.loonyparty.com/about/policy-proposals/": "entertainment",
                   "https://timdettmers.com/2023/01/30/which-gpu-for-deep-learning/": "data_science",
                   "https://gleam.run/cheatsheets/gleam-for-python-users/": "data_science",
                   "https://towardsdatascience.com/gpt-from-scratch-with-mlx-acf2defda30e": "data_science",
                   "https://blog.reedsy.com/short-story/a3gstd/": "entertainment",
                   "http://www.chakoteya.net/DoctorWho/40-1.html": "entertainment",
                   "https://stardewvalleywiki.com/Version_History": "gaming",
                   "https://alanwake.fandom.com/wiki/Alan_Wake_2": "gaming",
                   "https://www.polygon.com/23691206/best-fantasy-books-sci-fi-2023": "entertainment",
                   "https://arxiv.org/pdf/2404.10981": "data_science"
                   }

source_title_map = {"https://enterthegungeon.fandom.com/wiki/Bullet_Kin": "Bullet Kin",
                   "https://www.dropbox.com/scl/fi/ljtdg6eaucrbf1aksw5rm/c2%20-%20session%2050%20-%20underground.docx?rlkey=ioqwgkd14i5xk20i3fp38nzgs&e=1&dl=0": "The Paths through the Underground/Underdark",
                   "https://bytes-and-nibbles.web.app/bytes/stici-note-part-1-planning-and-prototyping": "Semantic and Textual Inference Chatbot Interface (STICI-Note)",
                   "https://github.com/llmware-ai/llmware": "LLMware",
                   "https://docs.marimo.io/recipes.html": "Building Block Recipes",
                   "https://towardsdatascience.com/how-to-maximize-your-impact-as-a-data-scientist-3881995a9cb1": "How to Maximize Your Impact as a Data Scientist",
                   "https://ec.europa.eu/commission/presscorner/detail/en/QANDA_21_1683": "Why do we need to regulate the use of Artificial Intelligence?",
                   "https://bg3.wiki/wiki/The_Emperor": "The Emperor",
                   "https://whattocook.substack.com/p/so-into-northern-spain": "So into northern spain!",
                   "https://dmtalkies.com/the-zone-of-interest-ending-explained-and-summary-2023-film/": "The Zone Of Interest’ Ending Explained & Film Summary: What Happens To Rudolf And Hedwig Hoss?",
                   "https://www.loonyparty.com/about/policy-proposals/": "Manifesto Proposals",
                   "https://timdettmers.com/2023/01/30/which-gpu-for-deep-learning/": "Which GPU(s) to Get for Deep Learning: My Experience and Advice for Using GPUs in Deep Learning",
                   "https://gleam.run/cheatsheets/gleam-for-python-users/": "Gleam for Python users",
                   "https://towardsdatascience.com/gpt-from-scratch-with-mlx-acf2defda30e": "GPT from Scratch with MLX",
                   "https://blog.reedsy.com/short-story/a3gstd/": "Galaxy Descriptions",
                   "http://www.chakoteya.net/DoctorWho/40-1.html": "Space Babies",
                   "https://stardewvalleywiki.com/Version_History": "The History of Stardew Valley",
                   "https://alanwake.fandom.com/wiki/Alan_Wake_2": "Alan Wake 2",
                   "https://www.polygon.com/23691206/best-fantasy-books-sci-fi-2023": "The Best Sci-Fi and Fantasy Books of 2023",
                   "https://arxiv.org/pdf/2404.10981": "A Survey on Retrieval-Augmented Text Generation for Large Language Models"
                   }

df_docs["source_type"] = df_docs["source_url"].map(source_type_map)
df_docs["title"] = df_docs["source_url"].map(source_title_map)

## Add text specific to a question

In [5]:

tct_add = ("There are several giants. One is burly, grey-skinned and 20 feet tall. "
           "It wears heavy dark iron "
           "armour covered in metal thorns and is leaning against a chamber wall. "
           "It is carrying "
           "what looks to be two tower shields nearly as tall as itself, both bearing "
           "menacing spikes. The other is taller and in more mobile irons, clutching a "
           "dangerous-looking maul the size of Yasha. It too is leaning against a wall and looking disinterested.")


df_docs.loc[1, "text"] = "{}\n\n{}".format(df_docs.loc[1, "text"],
                                           tct_add)
# df_docs.loc[1, "text"]


In [6]:
df_docs

,index,source_url,text,source_type,title
0,0,https://enterthegungeon.fandom.com/wiki/Bullet...,Bullet Kin\nBullet Kin are one of the most com...,gaming,Bullet Kin
1,1,https://www.dropbox.com/scl/fi/ljtdg6eaucrbf1a...,---The Paths through the Underground/Underdark...,gaming,The Paths through the Underground/Underdark
2,2,https://bytes-and-nibbles.web.app/bytes/stici-...,Semantic and Textual Inference Chatbot Interfa...,data_science,Semantic and Textual Inference Chatbot Interfa...
3,3,https://github.com/llmware-ai/llmware,llmware\n\nBuilding Enterprise RAG Pipelines w...,data_science,LLMware
4,4,https://docs.marimo.io/recipes.html,Recipes\nThis page includes code snippets or “...,recipes,Building Block Recipes
5,5,https://towardsdatascience.com/how-to-maximize...,How to Maximize Your Impact as a Data Scientis...,data_science,How to Maximize Your Impact as a Data Scientist
6,6,https://ec.europa.eu/commission/presscorner/de...,Why do we need to regulate the use of Artifici...,government,Why do we need to regulate the use of Artifici...
7,7,https://bg3.wiki/wiki/The_Emperor,The Emperor is a mind flayer who appears in Ba...,gaming,The Emperor
8,8,https://whattocook.substack.com/p/so-into-nort...,so into northern spain!\nour magical urban-plu...,recipes,So into northern spain!
9,9,https://dmtalkies.com/the-zone-of-interest-end...,‘The Zone Of Interest’ Ending Explained & Film...,entertainment,The Zone Of Interest’ Ending Explained & Film ...


## Create a passages dataframe

In [24]:
prows = []
pid = 1
for idx in df_docs.index:

    doc_index = "doc_{}".format(df_docs.loc[idx, "index"])
    raw_text = df_docs.loc[idx, "text"]

    blocks = raw_text.split('\n\n')
    for block in blocks:
        prows.append(dict(_id=pid,
                          doc_index=doc_index,
                          source_url=df_docs.loc[idx, "source_url"],
                          source_type=df_docs.loc[idx, "source_type"],
                          title=df_docs.loc[idx, "title"],
                          content=block
                          )
                     )
        pid += 1


df_pass = pd.DataFrame(data=prows)




## More clean up

In [5]:

# Add a document index column
df_docs["_id"] = df_docs["index"].apply(lambda x: "{}".format(x))
df_docs["doc_index"] = df_docs["index"].apply(lambda x: "doc_{}".format(x))
df_docs = df_docs.drop(columns="index")
df_docs.head()

df_docs["content"] = ""

# Step 1: Get text and metadata fields
content_max_len = 190000
for idx in df_docs.index:

    raw_text = df_docs.loc[idx, "text"]

    blocks = raw_text.split('\n\n')
    annotated_text = ""
    for i, block in enumerate(blocks, 1):
        annotated_text += f"[{i}]\n{block}\n\n"

    df_docs.loc[idx, "content"] = annotated_text[:content_max_len]

# Reduce and reorder columns
dfd_cols = [ '_id', 'doc_index', 'source_url', 'source_type', 'title', 'content']
df_docs = df_docs[dfd_cols]
df_docs


,_id,doc_index,source_url,source_type,title,content
0,0,doc_0,https://enterthegungeon.fandom.com/wiki/Bullet...,gaming,Bullet Kin,[1]\nBullet Kin\nBullet Kin are one of the mos...
1,1,doc_1,https://www.dropbox.com/scl/fi/ljtdg6eaucrbf1a...,gaming,The Paths through the Underground/Underdark,[1]\n---The Paths through the Underground/Unde...
2,2,doc_2,https://bytes-and-nibbles.web.app/bytes/stici-...,data_science,Semantic and Textual Inference Chatbot Interfa...,[1]\nSemantic and Textual Inference Chatbot In...
3,3,doc_3,https://github.com/llmware-ai/llmware,data_science,LLMware,[1]\nllmware\n\n[2]\nBuilding Enterprise RAG P...
4,4,doc_4,https://docs.marimo.io/recipes.html,recipes,Building Block Recipes,[1]\nRecipes\nThis page includes code snippets...
5,5,doc_5,https://towardsdatascience.com/how-to-maximize...,data_science,How to Maximize Your Impact as a Data Scientist,[1]\nHow to Maximize Your Impact as a Data Sci...
6,6,doc_6,https://ec.europa.eu/commission/presscorner/de...,government,Why do we need to regulate the use of Artifici...,[1]\nWhy do we need to regulate the use of Art...
7,7,doc_7,https://bg3.wiki/wiki/The_Emperor,gaming,The Emperor,[1]\nThe Emperor is a mind flayer who appears ...
8,8,doc_8,https://whattocook.substack.com/p/so-into-nort...,recipes,So into northern spain!,[1]\nso into northern spain!\nour magical urba...
9,9,doc_9,https://dmtalkies.com/the-zone-of-interest-end...,entertainment,The Zone Of Interest’ Ending Explained & Film ...,[1]\n‘The Zone Of Interest’ Ending Explained &...


In [6]:
df_docs.head(1).T
# print(df_docs.loc[0,"content"])

# df_docs.info()

# print(df_docs.loc[15, "content"])

df_docs.columns





Index(['_id', 'doc_index', 'source_url', 'source_type', 'title', 'content'], dtype='object')

In [7]:

# print("Source documents")
# display(df_docs.head())
# display(pd.DataFrame(df_docs["source_type"].value_counts()))
#
# print("Single-passage questions")
# display(df_spqs.head())


## Load documents to BigQuery

In [27]:
# Load dataframe to BigQuery
dataset_id = "ns_bq"
table_name = "rag_tests_3"
project_id = os.environ["GOOGLE_CLOUD_PROJECT_ID"]
df_load = df_pass

bqt.load_pandas_to_bigquery(df=df_load,
                            dataset_id=dataset_id,
                            table_name=table_name,
                            project_id=project_id,
                            if_exists="replace",
                            progress_bar_type="tqdm")


100%|██████████| 1/1 [00:00<00:00, 12052.60it/s]


## Create a view - Delete

In [13]:
from google.cloud import bigquery

def create_vais_view(project_id: str,
                     dataset_id: str,
                     table_id: str,
                     view_id: str):
    """
    Creates a BigQuery view that formats data for Vertex AI Search.
    """
    client = bigquery.Client(project=project_id)

    # Construct the full table and view paths
    source_table_path = f"{project_id}.{dataset_id}.{table_id}"
    view_path = f"{project_id}.{dataset_id}.{view_id}"

    # The SQL query to create the structural mapping
    sql = f"""
    CREATE OR REPLACE VIEW `{view_path}` AS
    SELECT
      _id AS id,
      STRUCT(
        content AS content,
        source_url AS source_url,
        source_type AS source_type,
        doc_index AS doc_index
      ) AS struct_data
    FROM `{source_table_path}`
    """

    print(f"Creating view: {view_path}...")
    query_job = client.query(sql)
    query_job.result()  # Wait for the job to complete

    print(f"✅ View created successfully. Use this as your 'bigquery_table' in the import function.")
    return view_path

In [14]:
view_name = "rag_tests_view"
create_vais_view(project_id=project_id,
                 dataset_id=dataset_id,
                 table_id=table_name,
                 view_id=view_name)


Creating view: ns-research-q4-2025.ns_bq.rag_tests_view...
✅ View created successfully. Use this as your 'bigquery_table' in the import function.


'ns-research-q4-2025.ns_bq.rag_tests_view'